# 전북도 고속도로 결빙 분석: 코드와 결과 확인

2026-09-23에 실행한 **데이터 감사 → 공간·시각 점검 → 개발용 EDA → 관측소 단위 검증 → 첫 ASOS 기준 실험**을 한곳에서 확인한다. 공식 최종 평가 RWIS 3개 지점(12410·33411·23402)의 정답 분포나 성능은 이 노트북에 포함하지 않는다.

노트북에는 원본 자료와 예측 행 출력이 저장돼 있지 않다. 아래 셀은 로컬의 Git 제외 경로 `outputs/private/`에 있는 집계 결과를 읽는다. 결과가 없으면 '전체 재생성' 셀의 스위치를 켜서 다시 만든다. 10분 결빙 평가 방식과 필수 추정 변수는 운영진 확인 전이므로 현재 점수는 **개발용 잠정 결과**다.

Update: Development-only 10-minute Ice predictions, provisional daily metrics, paired AWS-input comparison, and one-slot event context are now included below. The official held-out RWIS stations remain unevaluated.

In [ ]:
from pathlib import Path
import json
import os
import platform
import subprocess
import sys

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

ROOT = next(
    (p for p in [Path.cwd().resolve(), *Path.cwd().resolve().parents]
     if (p / "config" / "evaluation.json").exists() and (p / "scripts" / "audit_data.py").exists()),
    None,
)
if ROOT is None:
    raise FileNotFoundError("프로젝트 폴더 또는 notebooks/ 폴더에서 이 노트북을 열어주세요.")
PRIVATE = ROOT / "outputs" / "private"
CONFIG = json.loads((ROOT / "config" / "evaluation.json").read_text(encoding="utf-8"))
PROTECTED = set(CONFIG["held_out_rwis_stations"])
print("프로젝트:", ROOT)
print("Python:", platform.python_version(), "| 보호 RWIS:", ", ".join(sorted(PROTECTED)))

def read_json(relative):
    path = PRIVATE / relative
    if not path.exists():
        raise FileNotFoundError(f"{path} 없음. 아래 전체 재생성 셀을 실행하세요.")
    return json.loads(path.read_text(encoding="utf-8"))


## 실행 코드

| 단계 | 스크립트 | 확인 내용 |
| --- | --- | --- |
| 파일·변수 감사 | [audit_data.py](../scripts/audit_data.py) | ZIP 720개 CSV, 관측소·품질 |
| 날짜 경계 | [audit_timestamps.py](../scripts/audit_timestamps.py) | 빈 행, 자정 중복 |
| AWS 자정 중복 | [audit_aws_boundaries.py](../scripts/audit_aws_boundaries.py) | 수치 표기 동등성과 값 충돌 |
| AWS 입력 필터 | [audit_aws_cleaning.py](../scripts/audit_aws_cleaning.py), [weather_inputs.py](../src/weather_inputs.py) | 파일 날짜·KST·제외 건수 |
| 공간자료 | [audit_spatial.py](../scripts/audit_spatial.py) | 중심선·DEM·중첩 레이어 |
| 도로 위치 검증 | [audit_station_road_alignment.py](../scripts/audit_station_road_alignment.py) | 개발 RWIS·노선 중심선 거리 |
| 변수 사전 | [make_variable_dictionary.py](../scripts/make_variable_dictionary.py) | 실제 열·단위·형 |
| 개발용 EDA | [eda_development.py](../scripts/eda_development.py) | 보호 지점 제외 결빙 분포 |
| 분할 | [make_splits.py](../scripts/make_splits.py) | 외부 LOSO 14회·내부 3분할 |
| 첫 기준 모델 | [baseline_asos.py](../scripts/baseline_asos.py) | 과거 최근접 ASOS로 노면온도 추정 |
| 기상 특징 | [build_weather_features.py](../scripts/build_weather_features.py), [weather_features.py](../src/weather_features.py) | 과거 AWS·ASOS 시각 결합·연결률 |
| 기상 모델 | [baseline_weather_ridge.py](../scripts/baseline_weather_ridge.py) | 기상 융합 Ridge LOSO 점수 |
| 실패 진단 | [diagnose_weather_ridge.py](../scripts/diagnose_weather_ridge.py) | 겨울·AWS 연결 여부별 OOF 오차 |
| 시간 일반화 | [evaluate_weather_temporal.py](../scripts/evaluate_weather_temporal.py) | 첫 겨울 학습·둘째 겨울 보조 검증 |
| 평가 함수·테스트 | [validation.py](../src/validation.py), [test_validation.py](../tests/test_validation.py) | BIAS, 일별 지표 후보, 보호 지점 |

다음 셀은 테스트와 이틀짜리 작은 실행만 다시 수행한다. 전체 720개 파일의 감사나 240일 기준 모델은 자동 재실행하지 않는다.

In [ ]:
def captured_lines(stdout, stderr):
    text = "\n".join(part for part in (stdout, stderr) if part)
    return [line for line in text.splitlines() if line.strip()]


assert captured_lines(None, "ok") == ["ok"]
assert captured_lines("ok", None) == ["ok"]
assert captured_lines(None, None) == []

SMOKE_COMMANDS = [
    [sys.executable, "-m", "unittest", "discover", "-s", "tests", "-v"],
    [sys.executable, "scripts/audit_data.py", "--limit-files", "3"],
    [sys.executable, "scripts/baseline_asos.py", "--limit-days", "2"],
]
for command in SMOKE_COMMANDS:
    completed = subprocess.run(command, cwd=ROOT, text=True, capture_output=True, check=True)
    lines = captured_lines(completed.stdout, completed.stderr)
    print("OK", " ".join(command[1:]), " | ", lines[-1] if lines else "성공")


## 단계별 실행과 산출물 확인

아래 셀의 `RUN_STAGES`에 [실행 목록](../config/notebook_workflow.json)의 단계 ID를 넣고 실행한다. 예를 들어 `['baseline_asos']`는 ASOS 기준 실험만 다시 실행한다. `REBUILD_FULL=True`는 등록된 전체 단계를 순서대로 실행한다. 기본값은 재실행 없이 현재 로컬 산출물의 존재·수정 시각·선별 집계값을 확인한다. 실행 뒤 아래 결과 셀을 다시 실행하면 표와 그림이 갱신된다.

앞으로 새 단계는 스크립트와 요약 산출물을 만든 뒤 실행 목록에 등록하고, 필요한 해석 셀을 이 노트북에 추가한다. 제공 데이터 원본 행이나 OOF 예측 원자료는 노트북 출력으로 저장하지 않는다.

In [ ]:
WORKFLOW = json.loads((ROOT / "config" / "notebook_workflow.json").read_text(encoding="utf-8"))
assert WORKFLOW["schema_version"] == 1
STAGES = {stage["id"]: stage for stage in WORKFLOW["stages"]}
assert len(STAGES) == len(WORKFLOW["stages"])

# 필요한 단계만 선택해 실행한다. 전체 원본 스캔은 오래 걸리므로 기본값은 실행하지 않는다.
RUN_STAGES = []  # 예: ["weather_features", "weather_ridge"]
if os.environ.get("ICY_ROAD_RUN_STAGES"):
    RUN_STAGES = json.loads(os.environ["ICY_ROAD_RUN_STAGES"])
REBUILD_FULL = False
unknown = set(RUN_STAGES) - set(STAGES)
if unknown:
    raise ValueError(f"등록되지 않은 단계: {sorted(unknown)}")
selected = list(STAGES) if REBUILD_FULL else RUN_STAGES
for stage_id in selected:
    stage = STAGES[stage_id]
    command = [sys.executable, *stage["command"]]
    completed = subprocess.run(command, cwd=ROOT, text=True, capture_output=True, check=True)
    artifact = ROOT / stage["artifact"]
    if not artifact.exists():
        raise FileNotFoundError(f"{stage_id} 실행 후 산출물이 없습니다: {artifact}")
    print("OK", stage_id, "|", stage["artifact"])

status_rows = []
summary_rows = []
for stage in WORKFLOW["stages"]:
    artifact = ROOT / stage["artifact"]
    status_rows.append({
        "단계 ID": stage["id"],
        "설명": stage["name"],
        "산출물": stage["artifact"],
        "존재": artifact.exists(),
        "수정 시각 (KST)": pd.Timestamp(artifact.stat().st_mtime, unit="s", tz="UTC")
            .tz_convert("Asia/Seoul").strftime("%Y-%m-%d %H:%M") if artifact.exists() else None,
    })
    if artifact.suffix == ".json" and artifact.exists():
        result = json.loads(artifact.read_text(encoding="utf-8"))
        for key in stage.get("summary_keys", []):
            if key in result:
                value = result[key]
                summary_rows.append({"단계 ID": stage["id"], "지표": key,
                                     "값": json.dumps(value, ensure_ascii=False)})
display(pd.DataFrame(status_rows))
display(pd.DataFrame(summary_rows))


## 1. 파일·시각 품질

자료군별 행 수와 실측 지점을 확인한다. 빈 ID 행은 실측 지점 수에서 뺀다. AWS 자정 중복의 '원시 문자열 차이'는 숫자 값 차이로 확정되지 않았다.

In [ ]:
audit = read_json("full/audit_summary.json")
timing = read_json("full/timestamp_summary.json")
audit_rows = []
for kind in ("RWIS", "ASOS", "AWS"):
    ts = timing[kind]
    audit_rows.append({
        "자료": kind,
        "일별 파일": audit["file_counts"][kind],
        "행": audit["row_counts"][kind],
        "실측 지점": audit["station_counts"][kind] - int(ts["missing_id_rows"] > 0),
        "완전 빈 행": ts["empty_placeholder_rows"],
        "다음 날 00:00 행": ts["next_midnight_rows"],
        "다음 파일과 중복": ts["cross_file_midnight"].get("matched", 0),
    })
display(pd.DataFrame(audit_rows))
assert audit["file_counts"] == {"RWIS": 240, "ASOS": 240, "AWS": 240}
assert timing["AWS"]["empty_placeholder_rows"] == timing["AWS"]["missing_id_rows"]


### AWS 자정 중복과 입력 날짜 필터

같은 지점·같은 시각의 자정 행은 원시 문자열과 수치 정규화 값을 따로 비교했다. 원본 ZIP은 수정하지 않았다. AWS 모델 입력은 파일명 날짜의 KST 행만 유지한다. 이 단계는 관측값을 보간하거나 이상한 측정치를 복원하지 않는다.

In [ ]:
boundary = read_json("full/aws_boundary_audit.json")
cleaning = read_json("full/aws_cleaning_summary.json")
b = boundary["counts"]
c = cleaning["counts"]
display(pd.DataFrame([
    {"점검": "AWS 자정 중복", "수치": b["matched"]},
    {"점검": "수치 정규화 후 동일", "수치": b["equivalent_after_numeric_normalization"]},
    {"점검": "원시 문자열 동일", "수치": b["raw_identical"]},
    {"점검": "지점 없는 행 제외", "수치": c["missing_station_rows"]},
    {"점검": "파일명 날짜 밖 행 제외", "수치": c["valid_station_outside_nominal_day"]},
    {"점검": "유지한 AWS 행", "수치": c["retained_rows"]},
]))
print("짝이 없는 자정 행의 날짜:", boundary["unmatched_midnight_dates"])
assert b["matched"] == b["equivalent_after_numeric_normalization"]
assert b["matched"] == timing["AWS"]["cross_file_midnight"]["matched"]
assert not boundary["changed_columns"]
assert c["input_rows"] == audit["row_counts"]["AWS"]
assert c["missing_station_rows"] == timing["AWS"]["missing_id_rows"]
assert c["valid_station_outside_nominal_day"] == timing["AWS"]["next_midnight_rows"]
assert c["duplicate_station_time_rows"] == 0
assert c["input_rows"] == (c["missing_station_rows"] + c["valid_station_invalid_timestamp"]
                            + c["valid_station_outside_nominal_day"] + c["retained_rows"])


In [ ]:
spatial = read_json("spatial/spatial_summary.json")
road = spatial["road_centerline"]
dem = spatial["dem"]
display(pd.DataFrame([
    {"노선": route, "중심선 길이 km": round(length, 3)}
    for route, length in road["route_length_km"].items()
]))
print(f"중심선: {road['features']:,}개, CRS {road['crs']}, 무효 geometry {road['invalid_geometries']}개")
print(f"DEM: {dem['width']:,}×{dem['height']:,} 픽셀, 해상도 {dem['pixel_scale'][0]:.0f}m, NoData {dem['nodata']}")
print("DEM의 수직 고도 단위:", dem["vertical_unit_code"] or "메타데이터 미표기")

alignment = read_json("spatial/station_road_alignment_summary.json")
alignment_table = pd.read_csv(PRIVATE / "spatial" / "station_road_alignment.csv",
                              dtype={"station": str})
display(alignment_table[["station", "route", "distance_to_expected_m",
                         "nearest_matches_route"]].round(1))
assert alignment["stations"] == 14
assert alignment["nearest_route_mismatch"] == 0
assert not (set(alignment_table.station) & PROTECTED)


## 2. 개발용 결빙 분포와 검증 분할

분모는 유효 상태 코드가 있는 개발용 RWIS 14곳의 10분 관측이다. 보호 3곳의 동적 값은 포함하지 않는다. 연속된 Ice 시점은 사건 단위로도 센다.

In [ ]:
eda = read_json("development/eda_summary.json")
monthly = pd.read_csv(PRIVATE / "development" / "monthly_eda.csv")
stations = pd.read_csv(PRIVATE / "development" / "station_eda.csv", dtype={"station_id": str})
print(f"유효 상태 {eda['valid_label']:,}건 중 Ice {eda['ice_slots']:,}건 "
      f"({eda['ice_slots']/eda['valid_label']*100:.4f}%)")
print(f"Ice 사건 {eda['ice_events']}건, 단일 10분 사건 {eda['single_slot_events']}건")
display(stations[["station_id", "route", "valid_label", "ice_slots", "ice_events"]]
        .sort_values("ice_slots", ascending=False).reset_index(drop=True))
monthly["ice_per_10000"] = monthly["ice_slots"] / monthly["valid_label"] * 10000
display(monthly[["month", "valid_label", "ice_slots", "ice_per_10000"]])
fig, ax = plt.subplots(figsize=(9, 3))
ax.bar(monthly["month"], monthly["ice_per_10000"], color="#4477aa")
ax.set(ylabel="Ice slots / 10,000 valid slots", title="Development RWIS monthly Ice rate")
ax.tick_params(axis="x", rotation=45)
fig.tight_layout()
plt.show()
assert not (set(stations["station_id"]) & PROTECTED)


In [ ]:
splits = read_json("validation/splits.json")
outer = splits["outer_folds"]
for fold in outer:
    held = fold["validation_station"]
    train = set(fold["training_stations"])
    assert held not in train and not (train & PROTECTED)
    inner = [set(part["stations"]) for part in fold["inner_folds"]]
    assert set.union(*inner) == train
    assert all(not (a & b) for i, a in enumerate(inner) for b in inner[i + 1:])
display(pd.DataFrame([
    {"보류 RWIS": fold["validation_station"],
     "학습 지점": len(fold["training_stations"]),
     "내부 분할 양성 지점 수": [part["positive_stations"] for part in fold["inner_folds"]]}
    for fold in outer
]))
print("공간 외부 검증:", len(outer), "회 | 보호 지점 중첩: 0")


## 3. 과거 ASOS만 사용한 노면온도 기준 실험

각 목표 시각보다 **엄격히 이전**인 최근접 ASOS 기온(최대 60분 전)을 사용한다. 보정값은 해당 보류 RWIS를 제외한 13개 학습 지점의 중앙값이다. BIAS는 예측−관측이며 아래 표는 지점별 점수의 평균이다. 노면온도 단위와 ASOS 발표 지연은 확인이 필요하다.

In [ ]:
baseline = read_json("baseline/summary.json")
metrics = pd.read_csv(PRIVATE / "baseline" / "station_metrics.csv",
                      dtype={"station": str, "nearest_asos": str})
summary = pd.DataFrame([
    {"방법": "최근접 ASOS 그대로",
     "BIAS °C": baseline["direct_station_macro_bias"],
     "MAE °C": baseline["direct_station_macro_mae"],
     "RMSE °C": baseline["direct_station_macro_rmse"]},
    {"방법": "다른 지점의 중앙값 보정",
     "BIAS °C": baseline["adjusted_station_macro_bias"],
     "MAE °C": baseline["adjusted_station_macro_mae"],
     "RMSE °C": baseline["adjusted_station_macro_rmse"]},
])
display(summary.round(3))
display(metrics[["station", "nearest_asos", "distance_km", "direct_n",
                 "direct_mae", "adjusted_mae", "adjusted_rmse"]].round(3))
print("유효 쌍:", f"{baseline['total_valid_pairs']:,}/{baseline['total_targets']:,}",
      "| 보정 후 MAE 악화 지점:", baseline["offset_worsened_stations"])
fig, ax = plt.subplots(figsize=(7, 5))
ordered = metrics.sort_values("adjusted_mae").reset_index(drop=True)
y = np.arange(len(ordered))
for i, row in ordered.iterrows():
    ax.plot([row["direct_mae"], row["adjusted_mae"]], [i, i], color="#bbbbbb")
ax.scatter(ordered["direct_mae"], y, label="Direct ASOS", color="#4477aa", zorder=3)
ax.scatter(ordered["adjusted_mae"], y, label="LOSO adjusted", color="#cc6677", zorder=3)
ax.set(yticks=y, yticklabels=ordered["station"], xlabel="Road temperature MAE (°C)",
       ylabel="Held-out RWIS", title="Station-held-out baseline")
ax.legend()
fig.tight_layout()
plt.show()


In [ ]:
pred = pd.read_csv(PRIVATE / "baseline" / "predictions.csv",
                   usecols=["station", "time", "source_time", "age_minutes"],
                   dtype={"station": str})
target_time = pd.to_datetime(pred["time"])
source_time = pd.to_datetime(pred["source_time"])
valid = source_time.notna()
assert len(metrics) == len(splits["outer_folds"]) == 14
assert not (set(pred["station"]) & PROTECTED)
assert len(pred) == baseline["total_targets"]
assert int(valid.sum()) == baseline["total_valid_pairs"]
assert (source_time[valid] < target_time[valid]).all()
assert pred.loc[valid, "age_minutes"].between(10, 60).all()
assert np.isclose(metrics["direct_mae"].mean(), baseline["direct_station_macro_mae"])
assert np.isclose(metrics["adjusted_mae"].mean(), baseline["adjusted_station_macro_mae"])
print("보호 지점 제외, 과거 시각 입력, 10~60분 유효 범위, 지점 평균 점수 재계산: 모두 일치")


## 4. 과거 AWS·ASOS 기상 융합

원본 관측값을 화면에 나열하지 않고 연결률과 관측소별 OOF 오차만 표시한다. Ridge는 13개 학습 관측소에서만 결측 중앙값·스케일러를 맞추고 보류 관측소에 적용했다. 결과는 같은 ASOS 유효 시점에서 비교한다. 아래 −2~2°C 결과는 관측값으로 고른 **사후 진단 구간**이다.

In [ ]:
weather_features = read_json("weather_features/summary.json")
weather_model = read_json("weather_ridge/summary.json")
coverage = pd.read_csv(PRIVATE / "weather_features" / "station_coverage.csv", dtype={"station": str})
weather_metrics = pd.read_csv(PRIVATE / "weather_ridge" / "station_metrics.csv", dtype={"station": str})
print("기상 연결률: AWS", f"{weather_features['aws_temp_pairs']:,}/{weather_features['targets']:,}",
      "| ASOS", f"{weather_features['asos_temp_pairs']:,}/{weather_features['targets']:,}")
display(pd.DataFrame([
    {"방법": "ASOS 보정", "지점 평균 MAE °C": weather_model["asos_adjusted_station_macro_mae"]},
    {"방법": "과거 기상 융합 Ridge", "지점 평균 MAE °C": weather_model["weather_ridge_station_macro_mae"]},
]).round(3))
comparison = weather_metrics[["station", "n", "aws_available", "asos_adjusted_mae",
                              "weather_ridge_mae", "near_freezing_n",
                              "asos_adjusted_near_mae", "weather_ridge_near_mae"]].copy()
comparison["MAE 차이 °C"] = comparison.weather_ridge_mae - comparison.asos_adjusted_mae
display(comparison.round(3).sort_values("MAE 차이 °C"))
fig, ax = plt.subplots(figsize=(8, 4))
ordered = comparison.sort_values("MAE 차이 °C")
ax.barh(ordered.station, ordered["MAE 차이 °C"],
        color=np.where(ordered["MAE 차이 °C"] > 0, "#cc6677", "#4477aa"))
ax.axvline(0, color="black", linewidth=0.8)
ax.set(xlabel="Weather Ridge MAE − ASOS baseline MAE (°C)",
       title="Development station-held-out comparison")
fig.tight_layout()
plt.show()
assert weather_features["targets"] == baseline["total_targets"]
assert weather_features["asos_temp_pairs"] == baseline["total_valid_pairs"]
assert weather_model["paired_rows"] == baseline["total_valid_pairs"]
assert not (set(weather_metrics.station) & PROTECTED)
assert np.isclose(weather_metrics.asos_adjusted_mae.mean(),
                  weather_model["asos_adjusted_station_macro_mae"])
assert np.isclose(weather_metrics.weather_ridge_mae.mean(),
                  weather_model["weather_ridge_station_macro_mae"])
assert int((weather_metrics.weather_ridge_mae > weather_metrics.asos_adjusted_mae).sum()) == 1

diagnostic = read_json("weather_ridge/diagnostic_summary.json")
diagnostic_table = pd.read_csv(PRIVATE / "weather_ridge" / "winter_availability_diagnostic.csv",
                               dtype={"station": str})
print("전체 MAE 악화 지점:", diagnostic["weather_worse_stations"],
      "| −2~2°C 진단 구간 악화 지점:", diagnostic["near_freezing_worse_stations"])
display(diagnostic_table.loc[diagnostic_table.station.eq("12209"),
                             ["winter_start_year", "aws_available", "n",
                              "baseline_mae", "weather_mae",
                              "near_freezing_n", "baseline_near_mae", "weather_near_mae"]].round(3))
assert diagnostic["rows"] == weather_model["paired_rows"]


In [ ]:
feature_times = pd.read_parquet(PRIVATE / "weather_features" / "features.parquet",
                                columns=["station", "time", "aws_source_time", "asos_source_time"])
predicted = pd.read_parquet(PRIVATE / "weather_ridge" / "predictions.parquet",
                            columns=["station", "road_temp", "pred_weather_ridge"])
assert not (set(feature_times.station.astype(str)) & PROTECTED)
for name in ("aws_source_time", "asos_source_time"):
    present = feature_times[name].notna()
    assert (feature_times.loc[present, name] < feature_times.loc[present, "time"]).all()
errors = (predicted.pred_weather_ridge - predicted.road_temp).abs()
recomputed = errors.groupby(predicted.station.astype(str)).mean().sort_index()
reported = weather_metrics.set_index("station").weather_ridge_mae.sort_index()
assert np.allclose(recomputed.to_numpy(), reported.to_numpy())
print("보호 지점 제외, 미래·동시각 관측값 제외, OOF MAE 재계산: 모두 일치")


### 겨울 간 시간 일반화 보조 검증

첫 겨울 학습 후 둘째 겨울에 적용한다. **같은 RWIS 14곳이 학습과 검증에 모두 있으므로 새 도로 위치의 공간 일반화 점수와 합치지 않는다.** ASOS 보정 기준과 기상 Ridge를 둘째 겨울의 같은 유효 시점에서 비교한다.

In [ ]:
temporal = read_json("weather_temporal/summary.json")
temporal_stations = pd.read_csv(PRIVATE / "weather_temporal" / "station_metrics.csv",
                                dtype={"station": str})
display(pd.DataFrame([
    {"방법": "첫 겨울 ASOS 보정 → 둘째 겨울", "둘째 겨울 지점 평균 MAE °C": temporal["asos_adjusted_station_macro_mae"]},
    {"방법": "첫 겨울 기상 Ridge → 둘째 겨울", "둘째 겨울 지점 평균 MAE °C": temporal["weather_ridge_station_macro_mae"]},
]).round(3))
display(temporal_stations[["station", "n", "asos_adjusted_mae", "weather_ridge_mae"]].round(3))
assert temporal["status"] == "supplementary_temporal_transfer_only_not_spatial_holdout"
assert temporal["train_rows"] + temporal["validation_rows"] == weather_model["paired_rows"]
assert set(temporal_stations.station) == set(weather_metrics.station)
assert not (set(temporal_stations.station) & PROTECTED)
assert np.isclose(temporal_stations.weather_ridge_mae.mean(),
                  temporal["weather_ridge_station_macro_mae"])


## 해석과 다음 실행

ASOS 보정 기준 지점 평균 MAE 2.174°C에 비해 과거 AWS·ASOS 기상정보를 결합한 Ridge는 1.771°C였다. 이는 **개발용 14지점 공간 검증**이며 공식 3개소 점수나 조기예보 성능이 아니다. 첫 겨울 학습→둘째 겨울 보조 검증은 같은 지점에서 2.215→1.766°C였으며 공간 점수와 별도로 해석한다. 12209 지점과 −2~2°C 구간 일부 지점은 악화돼 후속 분석이 필요하다. 개발 RWIS 14곳은 제공 도로 중심선에서 모두 23m 이내였다. 다음에는 AWS 결측 대체 방식과 교량·터널·고도 같은 배포 가능한 정적 특징을 같은 분할에서 순차 비교한다.

세부 사항: [기상 결합 실험](../docs/WEATHER_FUSION_REPORT.md), [대회 요구사항](../docs/COMPETITION_REQUIREMENTS.md), [선행자료](../docs/REFERENCES.md), [검증 규약](../docs/VALIDATION_PROTOCOL.md), [운영진 질문](../docs/QUESTIONS_FOR_ORGANIZER.md), [현재 상태](../docs/PROJECT_STATUS.md).

### AWS fallback availability: official-reference follow-up case

The development-only audit checks the three nearest AWS stations using valid readings strictly before each RWIS time and at most 60 minutes old. It reports input coverage, not model skill. Read [the evidence and next-test matrix](../docs/FOLLOWUP_EVIDENCE.md) and [primary references](../docs/REFERENCES.md). No held-out RWIS labels are used.

In [ ]:
fallback = read_json("aws_fallback/summary.json")
assert fallback["status"] == "development_feature_availability_only"
assert fallback["station_count"] == 14
totals = fallback["totals"]
coverage = pd.DataFrame([
    {"input": "existing nearest AWS", "available": totals["existing_nearest_aws_times"]},
    {"input": "top 3 AWS with prior valid readings", "available": totals["top3_any_valid_times"]},
])
coverage["target_times"] = totals["target_times"]
coverage["coverage_percent"] = 100 * coverage["available"] / coverage["target_times"]
display(coverage)
display(pd.DataFrame([fallback["station_12209"]]))
print("Other AWS rescue:", totals["other_aws_rescue_times"],
      "| still missing:", totals["still_missing_times"])

station_coverage = pd.read_csv(PRIVATE / "aws_fallback" / "station_coverage.csv",
                               dtype={"rwis_station": str})
assert not set(station_coverage["rwis_station"]) & PROTECTED
station_coverage["existing_percent"] = (100 * station_coverage["existing_nearest_aws_times"]
                                        / station_coverage["target_times"]).round(1)
station_coverage["top3_percent"] = (100 * station_coverage["top3_any_valid_times"]
                                    / station_coverage["target_times"]).round(1)
display(station_coverage[["rwis_station", "existing_percent", "top3_percent",
                          "other_aws_rescue_times", "still_missing_times"]]
        .sort_values("other_aws_rescue_times", ascending=False))
aws_candidates = pd.read_csv(PRIVATE / "aws_fallback" / "nearest_aws.csv",
                             dtype={"rwis_station": str, "aws_station": str})
display(aws_candidates.loc[aws_candidates.rwis_station.eq("12209"),
                           ["aws_rank", "aws_station", "distance_km",
                            "valid_prior_times", "median_age_minutes_when_valid"]])


### Paired LOSO result: nearest versus fallback AWS

The same target rows, held-out stations, Ridge settings, and ASOS baseline are used for both variants. This is development evidence only. The [case matrix](../docs/FOLLOWUP_EVIDENCE.md) records failures and literature limits.

In [ ]:
variant = read_json("weather_fallback_comparison/summary.json")
assert variant["paired_rows"] == weather_model["paired_rows"]
display(pd.DataFrame([{"model": "nearest AWS", "macro_MAE_C": variant["nearest_macro_mae"],
                       "macro_RMSE_C": variant["nearest_macro_rmse"]},
                      {"model": "fallback AWS", "macro_MAE_C": variant["fallback_macro_mae"],
                       "macro_RMSE_C": variant["fallback_macro_rmse"]}]))
variant_station = pd.read_csv(PRIVATE / "weather_fallback_comparison" / "station_metrics.csv",
                              dtype={"station": str})
assert not set(variant_station.station) & PROTECTED
display(variant_station[["station", "n", "nearest_mae", "fallback_mae",
                         "near_freezing_n", "nearest_near_mae", "fallback_near_mae"]]
        .sort_values("station"))
print("Worse near freezing:", variant["stations_fallback_near_mae_worse"])
display(pd.DataFrame([variant["station_12209"]]))


## Competition evaluation scorecard and next runs

Read the Korean [evaluation status and execution order](../docs/EVALUATION_STATUS.md). The protected RWIS stations 12410, 33411, and 23402 remain reserved for one final test. Current numeric results cover development-site road-temperature estimation and Ice classification only. A development-only 10-minute Ice baseline and provisional daily scores now exist, but neither input variant is operationally adequate. No official daily score or three-route risk map has been produced. Next, examine one-slot Ice labels, prior moisture history, station-balanced threshold objectives, and spatial features using the same held-out-station folds.

## Development Ice classification: first complete baseline

This is a provisional evaluation on the 14 development RWIS stations. The three official held-out stations remain protected. The model uses only prior weather inputs; the stored score is not a calibrated probability. Read [the failure analysis and run commands](../docs/ICE_BASELINE_REPORT.md).

In [ ]:
ice_inputs = read_json("ice_features/summary.json")
ice_nearest = read_json("ice_logistic/summary.json")
ice_fallback = read_json("ice_fallback_logistic/summary.json")
ice_comparison = read_json("ice_variant_comparison/summary.json")
ice_diagnostic = read_json("ice_diagnostic/summary.json")
assert ice_inputs["valid_labels"] == ice_comparison["paired_valid_labels"] == 483652
assert ice_inputs["ice_slots"] == ice_comparison["ice_slots"] == 239
rows = []
for name, model in [("nearest AWS", ice_nearest), ("fallback AWS", ice_fallback)]:
    micro = model["slot_micro"]
    rows.append({"input": name, "TP": micro["tp"], "FP": micro["fp"],
                 "FN": micro["fn"], "precision": micro["precision"],
                 "recall": micro["recall"], "F1": micro["f1"],
                 "defined_daily_F1_days": model["daily_10minute_station_macro"]["defined_f1_days"]})
display(pd.DataFrame(rows))
display(pd.DataFrame(ice_comparison["fixed_observed_ice_days"]).T)
print("Accepted complete station-days:", ice_comparison["accepted_station_days"])
print("Ice AWS air-temperature coverage:",
      ice_diagnostic["ice_nearest_aws_temp_available"], "to",
      ice_diagnostic["ice_fallback_aws_temp_available"])


In [ ]:
ice_station = pd.read_csv(PRIVATE / "ice_variant_comparison" / "station_metrics.csv",
                          dtype={"station": str})
assert not set(ice_station.station) & PROTECTED
display(ice_station[["station", "ice_slots", "nearest_tp", "nearest_fp",
                     "fallback_tp", "fallback_fp"]].sort_values("station"))
print("Uncalibrated classifier scores and provisional daily policy; do not use as final score.")


### One-slot Ice event context audit

The development-only audit examines the immediately previous and next 10-minute road-state codes for each Ice run. It does not relabel or discard Ice. The event-level file is private; only aggregate counts are shown here.

In [ ]:
ice_events = read_json("ice_event_audit/summary.json")
assert ice_events["events"] == 215
assert ice_events["single_slot_events"] == 198
display(pd.DataFrame([ice_events]).T.rename(columns={0: "count_or_note"}))


## Facility proximity and road-temperature estimation

Two exploratory definitions are compared: distance to any supplied bridge/tunnel polygon, and distance only to polygons intersecting a 50 m buffer around the station's highway route. Both use the same held-out stations and timestamps. The source vintage during the observation winters is unverified.

In [ ]:
facility = read_json("spatial/station_facility/summary.json")
route_facility = read_json("spatial/station_facility_route/summary.json")
ablation = read_json("facility_weather_comparison/summary.json")
route_ablation = read_json("route_facility_weather_comparison/summary.json")
assert facility["station_count"] == route_facility["station_count"] == 14
assert ablation["paired_rows"] == route_ablation["paired_rows"] == read_json("aws_fallback_ridge/summary.json")["paired_rows"]
facility_station = pd.read_csv(PRIVATE / "route_facility_weather_comparison" / "station_metrics.csv", dtype={"station": str})
assert not set(facility_station.station) & PROTECTED
display(pd.DataFrame([{"variant": "weather only", "macro_MAE_C": ablation["weather_macro_mae"], "stations_improved": None}, {"variant": "any facility", "macro_MAE_C": ablation["facility_macro_mae"], "stations_improved": ablation["stations_facility_mae_better"]}, {"variant": "route-aligned facility", "macro_MAE_C": route_ablation["facility_macro_mae"], "stations_improved": route_ablation["stations_facility_mae_better"]}]))
display(facility_station[["station", "bridge_distance_m", "tunnel_distance_m", "weather_mae", "facility_mae", "weather_near_mae", "facility_near_mae"]].round(3))

## Prior three-hour AWS context and Ice

The 15-minute precipitation windows overlap, so their values are not added into a three-hour rainfall total. The table is descriptive, not causal; station, season and temporal dependence can confound the apparent association.

In [ ]:
history = read_json("ice_weather_history/summary.json")
e = history["exposure"]
assert history["ice_slots"] == 239
assert history["positive_history_available"] <= history["ice_slots"]
rate = pd.DataFrame([{"condition": "AWS air -5..3 C; any wet observation in prior 3h", "slots": e["cool_wet3h_rows"], "ice_slots": e["cool_wet3h_ice_slots"]}, {"condition": "AWS air -5..3 C; no wet observation in prior 3h", "slots": e["cool_dry3h_rows"], "ice_slots": e["cool_dry3h_ice_slots"]}])
rate["ice_fraction"] = rate["ice_slots"] / rate["slots"]
display(rate)
print("Positive Ice slots with prior AWS history:", history["positive_history_available"], "of", history["ice_slots"])

### Development LOSO classification with prior weather history

For each outer held-out station, the threshold is selected only from an inner station-level OOF prediction on the other stations. The class-weighted logistic score is not a calibrated probability. Daily metrics remain provisional pending organizer guidance.

In [ ]:
hist_model = read_json("ice_history_logistic/summary.json")
base_model = read_json("ice_fallback_logistic/summary.json")
paired_history = read_json("ice_history_comparison/summary.json")
assert hist_model["target_slots"] == base_model["target_slots"] == paired_history["paired_target_slots"]
assert hist_model["valid_label_slots"] == base_model["valid_label_slots"] == paired_history["paired_valid_labels"]
assert hist_model["ice_slots"] == base_model["ice_slots"] == 239
rows = []
for name, key in [("Current AWS", "fallback"), ("Plus 3h history", "history")]:
    m = paired_history[key + "_slot_micro"]
    fixed = paired_history["fixed_observed_ice_days"][key]
    rows.append({"model": name, "TP": m["tp"], "FP": m["fp"], "FN": m["fn"], "precision": m["precision"], "recall": m["recall"], "F1": m["f1"], "fixed_110_Ice_days_F1": fixed["mean_f1_equal_days"], "false_alarm_noIce_days": fixed["false_alarm_nonice_station_days"]})
display(pd.DataFrame(rows))
station_history = pd.read_csv(PRIVATE / "ice_history_comparison" / "station_metrics.csv", dtype={"station": str})
assert not set(station_history.station) & PROTECTED
display(station_history[["station", "ice_slots", "fallback_tp", "fallback_fp", "history_tp", "history_fp"]])

## Prior same-route RWIS benchmark

For each development target, take the nearest **other** development RWIS on the same highway route. Only road temperature measured strictly before the target, within 30 minutes, is usable. The three official protected RWIS are excluded from every source set. Compare the nonparametric neighbor value, the existing weather-only outer-LOSO Ridge, and their fixed 1:1 average on identical rows. This remains a development benchmark; source data latency and coverage on uninstrumented segments require separate checks.

In [ ]:
neighbor = read_json("neighbor_rwis/summary.json")
neighbor_diag = read_json("neighbor_rwis/diagnostic_summary.json")
neighbor_station = pd.read_csv(PRIVATE / "neighbor_rwis" / "station_metrics.csv", dtype={"station": str, "source_station": str})
assert neighbor["target_stations"] == 14
assert neighbor["paired_rows"] == neighbor_diag["paired_rows"]
assert not set(neighbor_station.station) & PROTECTED
assert not set(neighbor_station.source_station) & PROTECTED
assert not (neighbor_station.station == neighbor_station.source_station).any()
display(pd.DataFrame([{"input": "Prior AWS/ASOS Ridge", "macro_MAE_C": neighbor["weather_macro_mae_on_paired"], "macro_RMSE_C": neighbor["weather_macro_rmse_on_paired"], "near_freezing_MAE_C": neighbor["weather_macro_near_mae_on_paired"]}, {"input": "Prior nearest RWIS", "macro_MAE_C": neighbor["neighbor_macro_mae"], "macro_RMSE_C": neighbor["neighbor_macro_rmse"], "near_freezing_MAE_C": neighbor["neighbor_macro_near_mae"]}, {"input": "Fixed equal average", "macro_MAE_C": neighbor["equal_blend_macro_mae"], "macro_RMSE_C": neighbor["equal_blend_macro_rmse"], "near_freezing_MAE_C": neighbor["equal_blend_macro_near_mae"]}]))
display(neighbor_station[["station", "source_station", "source_distance_km", "paired_rows", "weather_mae", "neighbor_mae", "blend_mae"]].round(3))
print("Source observation age counts (minutes):", neighbor_diag["age_minutes_counts"])

In [ ]:
winter_rows = []
for name, values in neighbor_diag["winters"].items():
    winter_rows.append({"winter": name, **values})
display(pd.DataFrame(winter_rows))
route_rows = []
for name, values in neighbor_diag["routes"].items():
    route_rows.append({"route": name, **values})
display(pd.DataFrame(route_rows))
print("Both winter summaries stratify the same spatial-LOSO weather fits; they are not winter-forward validation.")

## 2026-10-02: 다음 겨울·미관측 지점 및 RWIS 지연 검증

구버전 20260915 개발 검증입니다. 공식 수정본 배포 후 재실행해야 합니다. 고정 1:1 혼합이며 보호 3개 지점은 사용하지 않습니다. OFFICIAL_UPDATE_20261002.md와 NEIGHBOR_TRANSFER_REPORT.md에서 공식 변경사항과 상세 설계를 확인하세요.


In [ ]:
transfer = read_json("neighbor_transfer/summary.json")
transfer_metrics = pd.read_csv(PRIVATE / "neighbor_transfer/station_metrics.csv", dtype={"station": str, "source_station": str})
folds = read_json("neighbor_transfer/fold_manifest.json")
assert len(folds) == 14
assert not set(transfer_metrics.station) & PROTECTED
assert not set(transfer_metrics.source_station) & PROTECTED
for fold in folds:
    assert fold["held_station"] not in fold["train_stations"]
    assert fold["held_station"] != fold["reference_station"]
    assert not set(fold["train_stations"]) & PROTECTED
    assert pd.Timestamp(fold["train_end"]) < pd.Timestamp(fold["valid_start"])
assert len({s["rows"] for s in transfer["scenarios"]}) == 1
display(pd.DataFrame(transfer["scenarios"]))
display(transfer_metrics[["station", "delay_minutes", "weather_mae", "blend_mae", "neighbor_available_rows"]])
print(transfer["limitations"])


## 비선형 결빙 분류 및 임계값 검증

동일 기상 입력·LOSO 분할의 개발 실험입니다. 내부 OOF에서만 임계값을 정하고 외부 지점으로 비교합니다. 모든 결과는 20260915 구버전이며 공식 최종 점수가 아닙니다. `ICE_NONLINEAR_REPORT.md`에 오경보·미탐과 채택 여부를 기록합니다. 모델 재학습은 시간이 걸리므로 기본 노트북은 저장된 결과만 확인합니다.

In [ ]:
nonlinear = read_json("ice_nonlinear/summary.json")
comparison = read_json("ice_nonlinear_comparison/summary.json")
assert nonlinear["target_rows"] == comparison["target_rows"] == 483805
assert comparison["valid_label_rows"] == 483652
assert comparison["ice_slots"] == 239
station_table = pd.read_csv(PRIVATE / "ice_nonlinear_comparison/station_metrics.csv", dtype={"station": str})
assert not set(station_table.station) & PROTECTED
for fold in read_json("ice_nonlinear/fold_manifest.json"):
    fit = set(fold["fit_stations"])
    check = set(fold["validation_stations"])
    assert not fit & check
    assert fold["outer_held"] not in fit | check
    assert not (fit | check) & PROTECTED
rows = []
for name, result in comparison["variants"].items():
    assert result["exact_episode_diagnostic"]["observed_events"] == 215
    rows.append({"variant": name, **result["slot_micro"],
                 "daily_station_macro_f1": result["daily_station_macro"]["f1"],
                 "fixed_positive_day_f1": result["fixed_ice_day_f1_station_first"],
                 "false_alarm_days": result["false_alarm_nonice_days"],
                 "detected_stations": result["stations_with_true_positives"]})
display(pd.DataFrame(rows))
display(station_table[["variant", "station", "positive_slots", "tp", "fp", "fn", "fixed_ice_day_f1"]])
print(comparison["fixed_positive_day_station_bootstrap"])
print(comparison["limitations"])


## 수정본 전 작업: 주변 관측과 도로 구간

PRE_CORRECTION_WORK_REPORT.md에 정보량·참조 결손·오경보·정적 구간 결과를 정리했습니다. outer_context는 감사용이며 모든 학습 분할에 공통 결합하면 안 됩니다. 구간 지도에는 위험 점수를 부여하지 않았습니다.

In [ ]:
context = read_json("rwis_context/summary.json")
assert context["target_rows"] == 483805 and context["ice_slots"] == 239
refs = pd.read_csv(PRIVATE / "rwis_context/inner_reference_audit.csv", dtype={"outer_held": str, "target": str, "source_station": str})
assert len(refs) == 1092
for row in refs.itertuples():
    if pd.notna(row.source_station):
        assert row.source_station not in set(row.excluded_stations.split(",")) | {row.target} | PROTECTED
coverage = pd.read_csv(PRIVATE / "rwis_context/station_coverage.csv", dtype={"station": str, "source_station": str, "moisture_station": str})
assert not set(coverage.station) & PROTECTED
assert not set(coverage.source_station) & PROTECTED
assert not set(coverage.moisture_station.dropna()) & PROTECTED
display(coverage)
print(context["fixed_latest_ice_benchmark"])
print(context["fixed_past60_ice_benchmark"])
print("Missing inner references:", context["inner_missing_reference_cases"])
display(pd.DataFrame(read_json("rwis_context/error_context_summary.json")["condition_groups"]))


In [ ]:
scaffold = read_json("spatial/segment_scaffold/summary.json")
assert scaffold["segments"] == 467
assert abs(scaffold["source_length_m"] - scaffold["segment_length_m"]) < .001
assert scaffold["roundtrip_gpkg_and_shapefile_verified"]
inventory = pd.read_csv(PRIVATE / "spatial/segment_scaffold/segment_inventory.csv")
assert inventory.seg_id.is_unique
assert inventory.risk_scr.isna().all()
assert inventory.risk_stat.eq("unscored").all()
assert inventory.length_m.between(0,1000.000001,inclusive="right").all()
display(pd.DataFrame(scaffold["routes"]))
print(scaffold["limitations"])
